In [2]:
from pyspark.sql import SparkSession

spark=SparkSession.builder.appName("app1").master("local[*]") \
     .config("spark.sql.adaptive.enabled", "false") \
      .getOrCreate()

df=spark.read.format("csv") \
   .option("inferSchema",True) \
    .option("header",True) \
    .load("orders.csv")

df.show(5)

+-------+----------+--------+--------+-------+-------------------+
|OrderID|  Customer|    Item|Quantity|  Price|          OrderDate|
+-------+----------+--------+--------+-------+-------------------+
|  O1001|      null| Monitor|       3| 1901.7|2025-08-01 11:11:00|
|  O1001|      null| Monitor|       3| 2000.7|2025-08-01 11:11:00|
|  O1002|Customer_2|Keyboard|       1|1314.52|2025-08-01 01:10:00|
|  O1003|Customer_2| Monitor|       1| 498.44|2025-08-01 02:27:00|
|  O1004|Customer_1|   Mouse|       2| 691.07|2025-08-01 00:59:00|
+-------+----------+--------+--------+-------+-------------------+
only showing top 5 rows



In [16]:
df.printSchema()

root
 |-- OrderID: string (nullable = true)
 |-- Customer: string (nullable = true)
 |-- Item: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Price: double (nullable = true)
 |-- OrderDate: timestamp (nullable = true)



In [17]:
from pyspark.sql.functions import * 
from pyspark.sql import Window

wind_spec=Window.partitionBy("customer","orderid","item",'price','orderdate').orderBy(col("orderdate").asc())
df_sales=df.withColumn('rank',row_number().over(wind_spec)).filter((col('item')=='Monitor') & (col("orderid")=='O1001'))
df_sales.show(5)

+-------+----------+-------+--------+------+-------------------+----+
|OrderID|  Customer|   Item|Quantity| Price|          OrderDate|rank|
+-------+----------+-------+--------+------+-------------------+----+
|  O1001|Customer_4|Monitor|       3|1901.7|2025-08-01 11:11:00|   1|
|  O1001|Customer_4|Monitor|       3|2000.7|2025-08-01 11:11:00|   1|
+-------+----------+-------+--------+------+-------------------+----+



In [3]:
import pyspark

print(pyspark.__version__)

3.5.8


In [18]:
# Q1. 

"""Given employer history data, where each record contains details about an employee’s 4
work history — employer, job position, and the start and end dates of each job.

We need to find out how many users had Microsoft as their employer, 
and immediately after that, they started working at Google, with no other 
employers between these two positions."""

# first create the dataframe
from pyspark.sql.functions import *
from pyspark.sql import Window

linkedin_data = [
    (1, 'Microsoft', 'developer', '2020-04-13', '2021-11-01'),
    (1, 'Google', 'developer', '2021-11-01', None),
    (2, 'Google', 'manager', '2021-01-01', '2021-01-11'),
    (2, 'Microsoft', 'manager', '2021-01-11', None),
    (3, 'Microsoft', 'analyst', '2019-03-15', '2020-07-24'),
    (3, 'Amazon', 'analyst', '2020-08-01', '2020-11-01'),
    (3, 'Google', 'senior analyst', '2020-11-01', '2021-03-04'),
    (4, 'Google', 'junior developer', '2018-06-01', '2021-11-01'),
    (4, 'Google', 'senior developer', '2021-11-01', None),
    (5, 'Microsoft', 'manager', '2017-09-26', None),
    (6, 'Google', 'CEO', '2015-10-02', None)
]

linkedn_schema= [
  'emp_id', 
  'employer', 
  'position', 
  'start_date', 
  'end_date'
]

employee_data=spark.createDataFrame(linkedin_data,schema=linkedn_schema)

window_spec= Window.partitionBy("emp_ID").orderBy(col("start_date").asc())
employee_switch=employee_data.withColumn("next_employee",lead("employer",1).over(window_spec))
final_employee_data=employee_switch.filter((lower(col("employer"))=='microsoft') & (lower(col("next_employee"))=='google'))
final_employee_data.show()
final_employee_data.explain(True)
num_partitions=final_employee_data.rdd.getNumPartitions()
print(num_partitions)

+------+---------+---------+----------+----------+-------------+
|emp_id| employer| position|start_date|  end_date|next_employee|
+------+---------+---------+----------+----------+-------------+
|     1|Microsoft|developer|2020-04-13|2021-11-01|       Google|
+------+---------+---------+----------+----------+-------------+

== Parsed Logical Plan ==
'Filter ((lower('employer) = microsoft) AND (lower('next_employee) = google))
+- Project [emp_id#527L, employer#528, position#529, start_date#530, end_date#531, next_employee#537]
   +- Project [emp_id#527L, employer#528, position#529, start_date#530, end_date#531, next_employee#537, next_employee#537]
      +- Window [lead(employer#528, 1, null) windowspecdefinition(emp_ID#527L, start_date#530 ASC NULLS FIRST, specifiedwindowframe(RowFrame, 1, 1)) AS next_employee#537], [emp_ID#527L], [start_date#530 ASC NULLS FIRST]
         +- Project [emp_id#527L, employer#528, position#529, start_date#530, end_date#531]
            +- LogicalRDD [emp_i

In [25]:
## Q1 using the spark sql
employee_data.createOrReplaceTempView("employee_table")

sql_df=spark.sql("select * from (select *, lead(employer) over(partition by emp_id order by end_date) as next_employer from employee_table) as " \
"tmp where employer='Microsoft' and next_employer='Google'")

sql_df.show()
sql_df.explain(True)

+------+---------+--------+----------+--------+-------------+
|emp_id| employer|position|start_date|end_date|next_employer|
+------+---------+--------+----------+--------+-------------+
|     2|Microsoft| manager|2021-01-11|    NULL|       Google|
+------+---------+--------+----------+--------+-------------+

== Parsed Logical Plan ==
'Project [*]
+- 'Filter (('employer = Microsoft) AND ('next_employer = Google))
   +- 'SubqueryAlias tmp
      +- 'Project [*, 'lead('employer) windowspecdefinition('emp_id, 'end_date ASC NULLS FIRST, unspecifiedframe$()) AS next_employer#692]
         +- 'UnresolvedRelation [employee_table], [], false

== Analyzed Logical Plan ==
emp_id: bigint, employer: string, position: string, start_date: string, end_date: string, next_employer: string
Project [emp_id#527L, employer#528, position#529, start_date#530, end_date#531, next_employer#692]
+- Filter ((employer#528 = Microsoft) AND (next_employer#692 = Google))
   +- SubqueryAlias tmp
      +- Project [emp_i

#### Question 2

In [ ]:
"""
Given a table of hotels with various attributes (hotel_address, 
additional_number_of_scoring, review_date, average_score, hotel_name, 
reviewer_nationality, negative_review, review_total_negative_word_counts, 
total_number_of_reviews, positive_review, review_total_positive_word_counts, 
total_number_of_reviews_reviewer_has_given, reviewer_score, tags, days_since_review, lat, lng ), 
We need to find the top 10 hotels with the highest average scores. The output should include:

The hotel name.
The average score of the hotel.
The records should be sorted by average score in descending order.

"""

# first create the dataframe 
data = [
  ('123 Ocean Ave, Miami, FL', 3, '2024-11-10', 4.2, 'Ocean View', 'American', 'Room small, but clean.', 5, 150, 'Great location and friendly staff!', 8, 30, 4.5, 'beachfront, family-friendly', '5 days', 25.7617, -80.1918),   
  ('456 Mountain Rd, Boulder, CO', 2, '2024-11-12', 3.9, 'Mountain Lodge', 'Canadian', 'wifi slow.', 3, 120, 'nice rooms.', 10, 20, 4.0, 'scenic, nature', '3 days', 40.015, -105.2705),  
  ('789 Downtown St, New York, NY', 5, '2024-11-15', 4.7, 'Central Park Hotel', 'British', 'Noisy, sleep.', 7, 200, 'Perfect location near Central Park.', 12, 50, 4.7, 'luxury, city-center', '1 day', 40.7831, -73.9712),
  ('101 Lakeside Blvd, Austin, TX', 1, '2024-11-08', 4.0, 'Lakeside Inn', 'Mexican', 'food avg.', 4, 80, 'Nice, friendly service.', 6, 15, 3.8, 'relaxing, family', '10 days', 30.2672, -97.7431),
  ('202 River Ave, Nashville, TN', 4, '2024-11-13', 4.5, 'Riverside', 'German', 'Limited parking', 2, 175, 'Great rooms.', 9, 25, 4.2, 'riverfront, peaceful', '2 days', 36.1627, -86.7816)
]
# Define columns for the hotel DataFrame
schema_columns = [
  "hotel_address", 
  "additional_number_of_scoring", 
  "review_date", 
  "customer_score", 
  "hotel_name",            
  "reviewer_nationality", 
  "negative_review", 
  "review_total_negative_word_counts", 
  "total_number_of_reviews",           
  "positive_review", 
  "review_total_positive_word_counts", 
  "total_number_of_reviews_reviewer_has_given",
  "reviewer_score", 
  "tags", 
  "days_since_review", 
  "lat", 
  "lng"
]
hotel_df=spark.createDataFrame(data,schema_columns)
hotel_df.show()



+--------------------+----------------------------+-----------+--------------+------------------+--------------------+--------------------+---------------------------------+-----------------------+--------------------+---------------------------------+------------------------------------------+--------------+--------------------+-----------------+-------+---------+
|       hotel_address|additional_number_of_scoring|review_date|customer_score|        hotel_name|reviewer_nationality|     negative_review|review_total_negative_word_counts|total_number_of_reviews|     positive_review|review_total_positive_word_counts|total_number_of_reviews_reviewer_has_given|reviewer_score|                tags|days_since_review|    lat|      lng|
+--------------------+----------------------------+-----------+--------------+------------------+--------------------+--------------------+---------------------------------+-----------------------+--------------------+---------------------------------+------------

In [32]:
## find out the avg score using the addtional_number_scoring+customer_score+reviewer_score/total_number_of_reviews_reviewer_has_given

hotel_review_score= hotel_df.withColumn("avg_score",round((col("additional_number_of_scoring")+col("customer_score")*col("reviewer_score"))/col("total_number_of_reviews_reviewer_has_given"),2))
hotel_review_score.show()

+--------------------+----------------------------+-----------+--------------+------------------+--------------------+--------------------+---------------------------------+-----------------------+--------------------+---------------------------------+------------------------------------------+--------------+--------------------+-----------------+-------+---------+---------+
|       hotel_address|additional_number_of_scoring|review_date|customer_score|        hotel_name|reviewer_nationality|     negative_review|review_total_negative_word_counts|total_number_of_reviews|     positive_review|review_total_positive_word_counts|total_number_of_reviews_reviewer_has_given|reviewer_score|                tags|days_since_review|    lat|      lng|avg_score|
+--------------------+----------------------------+-----------+--------------+------------------+--------------------+--------------------+---------------------------------+-----------------------+--------------------+--------------------------

In [ ]:
## highest avg score hotel
highest_review=hotel_review_score.orderBy(col('avg_score').desc())
highest_review.show()

+--------------------+----------------------------+-----------+--------------+------------------+--------------------+--------------------+---------------------------------+-----------------------+--------------------+---------------------------------+------------------------------------------+--------------+--------------------+-----------------+-------+---------+---------+
|       hotel_address|additional_number_of_scoring|review_date|customer_score|        hotel_name|reviewer_nationality|     negative_review|review_total_negative_word_counts|total_number_of_reviews|     positive_review|review_total_positive_word_counts|total_number_of_reviews_reviewer_has_given|reviewer_score|                tags|days_since_review|    lat|      lng|avg_score|
+--------------------+----------------------------+-----------+--------------+------------------+--------------------+--------------------+---------------------------------+-----------------------+--------------------+--------------------------

#### Question 3

In [2]:
"""
We have a table of employees that includes the following fields: id, first_name, 
last_name, age, sex, employee_title, department, salary, target, bonus, city, 
address, and manager_id. We need to find the top 3 distinct salaries for each department. 
The output should include:

The department name.
The top 3 distinct salaries for each department.
The results should be ordered alphabetically by department and then by the highest 
salary to the lowest salary.

"""

data = [
    (1, 'Allen', 'Wang', 55, 'F', 'Manager', 'Management', 200000, 0, 300, 'California', '23St', 1),
    (13, 'Katty', 'Bond', 56, 'F', 'Manager', 'Management', 150000, 0, 300, 'Arizona', None, 1),
    (19, 'George', 'Joe', 50, 'M', 'Manager', 'Management', 100000, 0, 300, 'Florida', '26St', 1),
    (11, 'Richerd', 'Gear', 57, 'M', 'Manager', 'Management', 250000, 0, 300, 'Alabama', None, 1),
    (10, 'Jennifer', 'Dion', 34, 'F', 'Sales', 'Sales', 100000, 200, 150, 'Alabama', None, 13),
    (18, 'Laila', 'Mark', 26, 'F', 'Sales', 'Sales', 100000, 200, 150, 'Florida', '23St', 11),
    (20, 'Sarrah', 'Bicky', 31, 'F', 'Senior Sales', 'Sales', 200000, 200, 150, 'Florida', '53St', 19),
    (21, 'Suzan', 'Lee', 34, 'F', 'Sales', 'Sales', 130000, 200, 150, 'Florida', '56St', 19),
    (22, 'Mandy', 'John', 31, 'F', 'Sales', 'Sales', 130000, 200, 150, 'Florida', '45St', 19),
    (17, 'Mick', 'Berry', 44, 'M', 'Senior Sales', 'Sales', 220000, 200, 150, 'Florida', None, 11),
    (12, 'Shandler', 'Bing', 23, 'M', 'Auditor', 'Audit', 110000, 200, 150, 'Arizona', None, 11),
    (14, 'Jason', 'Tom', 23, 'M', 'Auditor', 'Audit', 100000, 200, 150, 'Arizona', None, 11),
    (16, 'Celine', 'Anston', 27, 'F', 'Auditor', 'Audit', 100000, 200, 150, 'Colorado', None, 11),
    (15, 'Michale', 'Jackson', 44, 'F', 'Auditor', 'Audit', 70000, 150, 150, 'Colorado', None, 11),
    (6, 'Molly', 'Sam', 28, 'F', 'Sales', 'Sales', 140000, 100, 150, 'Arizona', '24St', 13),
    (7, 'Nicky', 'Bat', 33, 'F', 'Sales', 'Sales', None, None, None, None, None, None)
]
# Define columns for the employees DataFrame
columns = [
  "id", 
  "first_name", 
  "last_name", 
  "age", 
  "sex", 
  "employee_title", 
  "department", 
  "salary", 
  "target", 
  "bonus", "city", 
  "address", 
  "manager_id"
]

sales_df=spark.createDataFrame(data,columns)
sales_df.show()
sales_df.printSchema()

+---+----------+---------+---+---+--------------+----------+------+------+-----+----------+-------+----------+
| id|first_name|last_name|age|sex|employee_title|department|salary|target|bonus|      city|address|manager_id|
+---+----------+---------+---+---+--------------+----------+------+------+-----+----------+-------+----------+
|  1|     Allen|     Wang| 55|  F|       Manager|Management|200000|     0|  300|California|   23St|         1|
| 13|     Katty|     Bond| 56|  F|       Manager|Management|150000|     0|  300|   Arizona|   NULL|         1|
| 19|    George|      Joe| 50|  M|       Manager|Management|100000|     0|  300|   Florida|   26St|         1|
| 11|   Richerd|     Gear| 57|  M|       Manager|Management|250000|     0|  300|   Alabama|   NULL|         1|
| 10|  Jennifer|     Dion| 34|  F|         Sales|     Sales|100000|   200|  150|   Alabama|   NULL|        13|
| 18|     Laila|     Mark| 26|  F|         Sales|     Sales|100000|   200|  150|   Florida|   23St|        11|
|

In [7]:
## Group by employee by department id and then countDistinct salary, then sort the 
# department by asc, and salary by desc 
from pyspark.sql.functions import *
employee_salary_data=sales_df.select("department","salary").distinct()
group_data=employee_salary_data.orderBy("department",col("salary").desc())

group_data.show()

+----------+------+
|department|salary|
+----------+------+
|     Audit|110000|
|     Audit|100000|
|     Audit| 70000|
|Management|250000|
|Management|200000|
|Management|150000|
|Management|100000|
|     Sales|220000|
|     Sales|200000|
|     Sales|140000|
|     Sales|130000|
|     Sales|100000|
|     Sales|  NULL|
+----------+------+



In [10]:
## now you have to use the window function to partition by data 
from pyspark.sql import Window
dep_windows=Window.partitionBy(col("department")).orderBy("department",col("salary").desc())
employee_data_partition=employee_salary_data.withColumn("rnk",dense_rank().over(dep_windows)).filter("rnk<=3")
employee_data_partition.show()

+----------+------+---+
|department|salary|rnk|
+----------+------+---+
|     Audit|110000|  1|
|     Audit|100000|  2|
|     Audit| 70000|  3|
|     Sales|220000|  1|
|     Sales|200000|  2|
|     Sales|140000|  3|
|Management|250000|  1|
|Management|200000|  2|
|Management|150000|  3|
+----------+------+---+



### Salting

In [2]:
spark.conf.set("spark.sql.shuffle.partitions", "3")
spark.conf.get("spark.sql.shuffle.partitions")

'3'

In [3]:
## Skew Data frame
from pyspark.sql.types import IntegerType
df_uniform = spark.createDataFrame([i for i in range(1000000)], IntegerType())
df_uniform.show(5, False)

+-----+
|value|
+-----+
|0    |
|1    |
|2    |
|3    |
|4    |
+-----+
only showing top 5 rows



In [4]:
## apply the groupby to check how data is spread
from pyspark.sql.functions import * 

df_uniform.withColumn("partition",spark_partition_id()) \
.groupBy("partition").count().orderBy("partition").show()

+---------+-----+
|partition|count|
+---------+-----+
|        0|82944|
|        1|82944|
|        2|83968|
|        3|82944|
|        4|83968|
|        5|82944|
|        6|82944|
|        7|83968|
|        8|82944|
|        9|83968|
|       10|82944|
|       11|83520|
+---------+-----+



In [5]:
## Create the another dataset with unever distribution
df0 = spark.createDataFrame([0] * 999990, IntegerType()).repartition(1)
df1 = spark.createDataFrame([1] * 15, IntegerType()).repartition(1)
df2 = spark.createDataFrame([2] * 10, IntegerType()).repartition(1)
df3 = spark.createDataFrame([3] * 5, IntegerType()).repartition(1)
df_skew = df0.union(df1).union(df2).union(df3)


In [ ]:
df_skew.show(5)

+-----+
|value|
+-----+
|    0|
|    0|
|    0|
|    0|
|    0|
+-----+
only showing top 5 rows



In [6]:
df_join=df_uniform.join(df_skew,"value","inner")
df_join.show(5)

+-----+
|value|
+-----+
|    0|
|    0|
|    0|
|    0|
|    0|
+-----+
only showing top 5 rows



In [8]:
df_join.withColumn("partition", spark_partition_id()) \
.groupBy("partition").count().show(5)

+---------+-------+
|partition|  count|
+---------+-------+
|        0|1000005|
|        1|     15|
+---------+-------+



### Adding Salting

In [9]:
## add the salt column in first dataframe

SALT_NUMBER = int(spark.conf.get("spark.sql.shuffle.partitions"))
SALT_NUMBER

3

In [10]:
df_skew = df_skew.withColumn("salt", (rand() * SALT_NUMBER).cast("int"))
df_skew.show(10, truncate=False)

+-----+----+
|value|salt|
+-----+----+
|0    |1   |
|0    |2   |
|0    |1   |
|0    |0   |
|0    |1   |
|0    |0   |
|0    |0   |
|0    |2   |
|0    |1   |
|0    |0   |
+-----+----+
only showing top 10 rows



In [11]:
## adding the lit array in second dataframe and then explode the salt column

df_uniform = (
    df_uniform
    .withColumn("salt_values", array([lit(i) for i in range(SALT_NUMBER)]))
    .withColumn("salt", explode(col("salt_values")))
)

df_uniform.show(10, truncate=False)

+-----+-----------+----+
|value|salt_values|salt|
+-----+-----------+----+
|0    |[0, 1, 2]  |0   |
|0    |[0, 1, 2]  |1   |
|0    |[0, 1, 2]  |2   |
|1    |[0, 1, 2]  |0   |
|1    |[0, 1, 2]  |1   |
|1    |[0, 1, 2]  |2   |
|2    |[0, 1, 2]  |0   |
|2    |[0, 1, 2]  |1   |
|2    |[0, 1, 2]  |2   |
|3    |[0, 1, 2]  |0   |
+-----+-----------+----+
only showing top 10 rows



In [12]:
## Now join based on the salt and value

df_join_new=df_skew.join(df_uniform, ["value", "salt"], 'inner')
df_join_new.show()

+-----+----+-----------+
|value|salt|salt_values|
+-----+----+-----------+
|    0|   1|  [0, 1, 2]|
|    0|   1|  [0, 1, 2]|
|    0|   1|  [0, 1, 2]|
|    0|   1|  [0, 1, 2]|
|    0|   1|  [0, 1, 2]|
|    0|   1|  [0, 1, 2]|
|    0|   1|  [0, 1, 2]|
|    0|   1|  [0, 1, 2]|
|    0|   1|  [0, 1, 2]|
|    0|   1|  [0, 1, 2]|
|    0|   1|  [0, 1, 2]|
|    0|   1|  [0, 1, 2]|
|    0|   1|  [0, 1, 2]|
|    0|   1|  [0, 1, 2]|
|    0|   1|  [0, 1, 2]|
|    0|   1|  [0, 1, 2]|
|    0|   1|  [0, 1, 2]|
|    0|   1|  [0, 1, 2]|
|    0|   1|  [0, 1, 2]|
|    0|   1|  [0, 1, 2]|
+-----+----+-----------+
only showing top 20 rows



In [13]:
(
    df_join_new
    .withColumn("partition",spark_partition_id())
    .groupBy("value", "partition")
    .count()
    .orderBy("value", "partition")
    .show()
)

+-----+---------+------+
|value|partition| count|
+-----+---------+------+
|    0|        0|333506|
|    0|        1|333613|
|    0|        2|332871|
|    1|        0|     8|
|    1|        1|     7|
|    2|        0|     1|
|    2|        1|     7|
|    2|        2|     2|
|    3|        0|     1|
|    3|        1|     2|
|    3|        2|     2|
+-----+---------+------+

